In [ ]:
import torch
import numpy as np
import os
from transformers import AutoModelForCausalLM

model_name = "roneneldan/TinyStories-1M"
print(f"Loading {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name)
state_dict = model.state_dict()

FRACTIONAL_BITS = 8
SCALE_FACTOR = 2 ** FRACTIONAL_BITS
MIN_VAL = -32768
MAX_VAL = 32767

def float_to_q88(tensor):
    q_tensor = torch.round(tensor * SCALE_FACTOR)
    q_tensor = torch.clamp(q_tensor, MIN_VAL, MAX_VAL)
    return q_tensor.to(torch.int16)

def q88_to_float(q_tensor):
    return q_tensor.to(torch.float32) / SCALE_FACTOR

def export_to_hex_file(q_tensor, filename):
    flat_q = q_tensor.flatten().numpy()
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        for val in flat_q:
            f.write(f"{int(val) & 0xFFFF:04X}\n")

out_dir = "tinystories_q88_roms"
print(f"\nExporting weights to '{out_dir}/' ...")

for name, param in state_dict.items():
    q_param = float_to_q88(param)
    safe_name = name.replace('.', '_') + ".hex"
    export_to_hex_file(q_param, os.path.join(out_dir, safe_name))

print(f"Successfully exported {len(state_dict)} tensors to hex files.")

print("\n--- Quantization Error Analysis ---")

test_layer_name = None
test_weight = None

for name, param in state_dict.items():
    if len(param.shape) == 2:
        test_layer_name = name
        test_weight = param
        break

if test_weight is not None:
    print(f"Evaluating Layer: {test_layer_name}")
    print(f"Layer Shape: {test_weight.shape}")

    in_features = test_weight.shape[1]

    num_samples = 15
    sample_inputs = torch.randn(num_samples, in_features)

    fp32_output = torch.matmul(sample_inputs, test_weight.t())
    q_weight = float_to_q88(test_weight)
    dq_weight = q88_to_float(q_weight)
    q_output_simulated = torch.matmul(sample_inputs, dq_weight.t())


    absolute_errors = torch.abs(fp32_output - q_output_simulated)
    mae = torch.mean(absolute_errors).item()
    max_error = torch.max(absolute_errors).item()

    print(f"Original Weight Range: [{torch.min(test_weight):.4f}, {torch.max(test_weight):.4f}]")
    print(f"Mean Absolute Error (MAE): {mae:.6f}")
    print(f"Max Absolute Error:      {max_error:.6f}")
else:
    print("Could not find a 2D linear layer for testing.")

import shutil
shutil.make_archive("tinystories_q88_roms", 'zip', out_dir)


Loading roneneldan/TinyStories-1M...


Loading weights:   0%|          | 0/108 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-1M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Exporting weights to 'tinystories_q88_roms/' ...
Successfully exported 109 tensors to hex files.

--- Quantization Error Analysis ---
Evaluating Layer: transformer.wte.weight
Layer Shape: torch.Size([50257, 64])
Original Weight Range: [-0.6169, 0.8020]
Mean Absolute Error (MAE): 0.007066
Max Absolute Error:      0.041206

Created 'tinystories_q88_roms.zip' for download.


In [ ]:
print(model)

In [ ]:
print([f"{int(x) & 0xFFFF:04X}" for x in q_param.flatten()[:5]])

['FFDF', '001C', 'FFFE', 'FFF6', 'FFEE']


In [ ]:
sd = model.state_dict()
print(list(sd.keys())[:10])  # confirm keys

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/108 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-1M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['transformer.wte.weight', 'transformer.wpe.weight', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.attn.attention.k_proj.weight', 'transformer.h.0.attn.attention.v_proj.weight', 'transformer.h.0.attn.attention.q_proj.weight', 'transformer.h.0.attn.attention.out_proj.weight', 'transformer.h.0.attn.attention.out_proj.bias', 'transformer.h.0.ln_2.weight']


In [4]:
model_id = "roneneldan/TinyStories-1M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model.eval()

os.makedirs("golden_data", exist_ok=True)

activations = {}
def get_activation(name):
    def hook(model, input, output):

        hidden_states = output[0] if isinstance(output, tuple) else output

        activations[name] = hidden_states[0, -1, :].detach().numpy()
    return hook
model.transformer.h[0].register_forward_hook(get_activation('l0'))
model.transformer.h[1].register_forward_hook(get_activation('l1'))
model.transformer.h[2].register_forward_hook(get_activation('l2'))
model.transformer.h[3].register_forward_hook(get_activation('l3'))
model.transformer.ln_f.register_forward_hook(get_activation('final_norm'))

# Using 6 random token IDs as starting prompts
prompts = [12, 45, 102, 300, 345, 678]

print("Generating Golden Data...")
for i, prompt_id in enumerate(prompts):
    inputs = torch.tensor([[prompt_id]])


    with torch.no_grad():
        _ = model(inputs)

    np.save(f"golden_data/p{i}_l0.npy", activations['l0'])
    np.save(f"golden_data/p{i}_l1.npy", activations['l1'])
    np.save(f"golden_data/p{i}_l2.npy", activations['l2'])
    np.save(f"golden_data/p{i}_l3.npy", activations['l3'])
    np.save(f"golden_data/p{i}_norm.npy", activations['final_norm'])


    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=27, do_sample=False, pad_token_id=tokenizer.eos_token_id)


    sequence = outputs[0].numpy()
    np.savetxt(f"golden_data/p{i}_tokens.txt", sequence, fmt='%d')

shutil.make_archive("golden_data", 'zip', "golden_data")

Loading weights:   0%|          | 0/108 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-1M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating Golden Data...


'/content/golden_data.zip'